# multilingual-e5-small 문서 임베딩 (Colab GPU)

`scripts/compare_base_embeddings.py`가 쓸 e5-small 문서 벡터를 GPU로 만든다. 로컬 CPU에서는 기본 ko-sroberta(128토큰)만 71분이 걸렸다.

- 런타임: GPU(T4 이상)
- HF 토큰: Colab 왼쪽 열쇠 아이콘(Secrets)에 `HF_TOKEN`을 넣고 노트북 접근을 켠다. 없으면 실행 중에 입력을 받는다.
- 결과: 비공개 데이터셋 `blanden77/dograg-data`의 `experiments` 브랜치에 `e5/base_embeddings_e5-small_rows.npy`로 올린다(문서 본문은 올리지 않음). 행 순서 = `df.csv` 행 번호 = 앱의 문서 ID.

In [1]:
!pip -q install -U sentence-transformers huggingface_hub
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU 없음: 런타임 유형을 GPU로 바꾸세요')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 130.0 MB/s eta 0:00:00
Tesla T4


In [ ]:
import getpass
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
HF_TOKEN = (HF_TOKEN or getpass.getpass('HF_TOKEN (영문 입력 상태에서 붙여 넣기): ')).strip()
# A token typed with the Korean input mode on turns into Hangul and fails later as UnicodeEncodeError.
if not HF_TOKEN.isascii() or not HF_TOKEN.startswith('hf_'):
    raise ValueError(f'HF_TOKEN이 hf_로 시작하는 영문·숫자 값이 아닙니다(길이 {len(HF_TOKEN)}, 한글 포함: {not HF_TOKEN.isascii()}). '
                     'Hugging Face 토큰을 복사해 Colab Secrets의 HF_TOKEN에 붙여 넣으세요.')
print('HF_TOKEN ok, length', len(HF_TOKEN))

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
REPO, REVISION = 'blanden77/dograg-data', 'data-v3'  # src/private_data.py
path = hf_hub_download(REPO, 'data/df.csv', repo_type='dataset', revision=REVISION, token=HF_TOKEN)
df = pd.read_csv(path)
# The app's documents are qa.input with carriage returns removed (checked: 19,206 of 19,206 match the Chroma texts).
CR = chr(13)
texts = df['qa.input'].astype(str).str.replace(CR, '', regex=False).tolist()
print(len(texts))

In [ ]:
import time, numpy as np
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('intfloat/multilingual-e5-small', device='cuda')
model.max_seq_length = 512
started = time.time()
vectors = model.encode(['passage: ' + t for t in texts], batch_size=128, normalize_embeddings=True,
                       show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
print(vectors.shape, f'{time.time() - started:.0f}s')
np.save('base_embeddings_e5-small_rows.npy', vectors)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
api.create_branch(REPO, repo_type='dataset', branch='experiments', exist_ok=True)
api.upload_file(path_or_fileobj='base_embeddings_e5-small_rows.npy', path_in_repo='e5/base_embeddings_e5-small_rows.npy',
                repo_id=REPO, repo_type='dataset', revision='experiments')
print('uploaded to', REPO, '@ experiments')